# Synthesis Tutorial in native FlagQuantum

Complete local computational Notebook workflow: the same native arithmetic program synthesized under both width/depth objectives, text drawing and actual width/depth reports. Independent Circuit acceptance checks both objectives; execution and named readout additionally verify the result. No global-optimum or guaranteed resource improvement is claimed.

Define the native quantum functions below, synthesize them locally, inspect the executed results, and compare the complete complex state with an independent Circuit.

See [the native Python entry](main.py).

## Native imports for main

In [ ]:
from typing import Literal
import flagquantum as fq
from flagquantum import synth
from flagquantum.compiler import estimate_resources
from flagquantum.drawer import draw
from flagquantum.synth import Output, QNum, allocate, hadamard_transform, qfunc

## main

In [ ]:
@qfunc
def main(x: Output[QNum], y: Output[QNum]) -> None:
    allocate(3, x)
    hadamard_transform(x)
    y |= x**2 + 1

## build program

In [ ]:
def build_program(objective: Literal["width", "depth"]) -> synth.Program:
    return synth.synthesize(main, objective=objective, memory_limit_bytes=512 * 1024**2)

## run example

In [ ]:
def run_example() -> None:
    objectives: tuple[Literal["width", "depth"], ...] = ("depth", "width")
    for objective in objectives:
        program = build_program(objective)
        print(draw(program.circuit))
        resources = estimate_resources(program.circuit)
        print(
            objective,
            "resources:",
            {"width": resources.n_qubits, "depth": resources.depth},
        )
        result = fq.run(program)
        print(dict(program.probabilities(result, registers=("x", "y"))))

## Execute the complete native workflow

Run every cell in order. Any sampling is local and seeded; resource reports come from the actual native program. Read the reported approximation limits and optimization outcomes alongside the numerical results.

In [ ]:
run_example()

## Verify absolute amplitudes and clean workspace

The independent Circuit uses the same declared register order. The comparison checks every complex amplitude without global-phase alignment and verifies zero probability in hidden work qubits. This is a local numerical check.


In [ ]:
import torch
from flagquantum import synth
from examples.synthesis.compare_states import compare_states
from examples.synthesis.tutorials.basics.quantum_programming.synthesis_tutorial.comparisons.circuit import (
    reference_circuit,
)

for objective in ("width", "depth"):
    compare_states(build_program(objective), reference_circuit(), ("x", "y"))
print("Independent full-state comparison: passed")
